# Aquarium Detection — object detection, step by step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s6-aquarium-detection.ipynb)

Find every animal in an aquarium photo: a **bounding box** and a **class** for each, among
**7 classes** (fish, jellyfish, penguin, puffin, shark, starfish, stingray):
[challenge 195](https://ml-arena.com/viewchallenge/195) on ML-Arena. Metric: **mAP50-95**.

| Section | What you do |
|---|---|
| 0–3 | Download the data, look at it, hold out a validation set |
| 4 | **Data analysis**: classes, box sizes, boxes per image, overlaps |
| 5 | **A small YOLO from scratch**: grid, boxes per cell, anchors, loss — first submission |
| 6 | **Pretrained detectors as they are**: what do they see in an aquarium? |
| 7 | **Fine-tune one** — second submission |

> In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
> The only thing you must edit is your API token (your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk>=3" ultralytics

import mlarena
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random, shutil, sys, zipfile, torch
from pathlib import Path
from PIL import Image

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
CHALLENGE_ID = 195

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the data

`train.zip` holds 448 images and their labels, `test_images.zip` the 190 images you predict on.
`detection_metric.py` is the leaderboard's parser and metric: we import it to score locally.

In [ ]:
client.download_dataset(CHALLENGE_ID, "data/")
zipfile.ZipFile("data/train.zip").extractall("data/train")
zipfile.ZipFile("data/test_images.zip").extractall("data/test")
sys.path.insert(0, "data")
import detection_metric as dm

class_names = pd.read_csv("data/classes.csv")["name"].tolist()
train_ids = sorted(p.stem for p in Path("data/train/images").glob("*.jpg"))
test_images = sorted(Path("data/test/images").glob("*.jpg"))
print(len(train_ids), "training images,", len(test_images), "test images")
print(class_names)

## 2. Look at the boxes

A label file has one line per box: `class_id x_center y_center width height`, each coordinate
divided by the image's width or height (the **YOLO format**).

In [ ]:
def read_labels(image_id):
    lines = Path(f"data/train/labels/{image_id}.txt").read_text().split()
    return np.array(lines, dtype=float).reshape(-1, 5)

def draw(ax, image_path, boxes, title=""):
    """Draw labels (class x y w h) or predictions (class confidence x y w h) on an image."""
    img = Image.open(image_path)
    W, H = img.size
    ax.imshow(img)
    for cls, xc, yc, w, h in boxes[:, [0, -4, -3, -2, -1]]:
        color = plt.cm.tab10(int(cls))
        ax.add_patch(patches.Rectangle(((xc - w / 2) * W, (yc - h / 2) * H), w * W, h * H,
                                       linewidth=2, edgecolor=color, facecolor="none"))
        ax.text((xc - w / 2) * W, (yc - h / 2) * H, class_names[int(cls)], color="white",
                fontsize=7, bbox=dict(facecolor=color, pad=1, linewidth=0))
    ax.set_title(title, fontsize=9); ax.axis("off")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, iid in zip(axes.flat, random.Random(0).sample(train_ids, 6)):
    boxes = read_labels(iid)
    draw(ax, f"data/train/images/{iid}.jpg", boxes, f"{len(boxes)} boxes")
plt.tight_layout(); plt.show()

## 3. Hold out a validation set

The test labels are private, so we keep 15 % of the training images aside to measure the model
before submitting, and write the `data.yaml` Ultralytics trains from.

In [ ]:
ids = train_ids[:]
random.Random(0).shuffle(ids)
split = {"val": ids[:len(ids) * 15 // 100], "train": ids[len(ids) * 15 // 100:]}
for name, part in split.items():
    for sub in ("images", "labels"):
        Path(f"yolo/{name}/{sub}").mkdir(parents=True, exist_ok=True)
    for iid in part:
        shutil.copy(f"data/train/images/{iid}.jpg", f"yolo/{name}/images/")
        shutil.copy(f"data/train/labels/{iid}.txt", f"yolo/{name}/labels/")
Path("aquarium.yaml").write_text(
    f"path: {Path('yolo').resolve()}\ntrain: train/images\nval: val/images\nnames: {class_names}\n")
print({k: len(v) for k, v in split.items()})

## 4. Data analysis

Before any model: what does a detector have to cope with here? One table, one row per
training box (the validation images stay unseen). Sizes are fractions of the image side.

In [ ]:
boxes = pd.DataFrame([(iid, *box) for iid in split["train"] for box in read_labels(iid)],
                     columns=["image_id", "class_id", "x", "y", "w", "h"])
boxes["class"] = boxes["class_id"].astype(int).map(dict(enumerate(class_names)))
boxes["size"] = np.sqrt(boxes["w"] * boxes["h"])         # side of a square of the same area
boxes["size_bin"] = pd.cut(boxes["size"], [0, 0.05, 0.15, 1], labels=["small", "medium", "large"])
print(len(boxes), "boxes in", boxes["image_id"].nunique(), "images")

boxes["w_over_h"] = boxes["w"] / boxes["h"]
boxes.groupby("class").agg(boxes=("x", "size"), images=("image_id", "nunique"),
                           median_size=("size", "median"), median_w_over_h=("w_over_h", "median"))\
     .sort_values("boxes", ascending=False).round(2)

In [ ]:
per_image = boxes.groupby("image_id").size()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
per_image.plot.hist(bins=40, ax=axes[0], title=f"boxes per image (median {per_image.median():.0f}, max {per_image.max()})")
boxes["size"].plot.hist(bins=60, ax=axes[1], title="box size (fraction of the image side)")
for name, g in boxes.groupby("class"):
    axes[2].scatter(g["w"], g["h"], s=3, label=name)
axes[2].set(xlabel="width", ylabel="height", title="box shapes"); axes[2].legend(markerscale=4, fontsize=7)
plt.tight_layout(); plt.show()
print(boxes["size_bin"].value_counts(normalize=True).round(2).to_dict(), "(small: under 5 % of the side, about 32 px at 640)")

Do boxes overlap? For every box, its largest IoU with another box of the same image.

In [ ]:
def max_overlap(labels):
    iou = dm.box_iou(labels[:, 1:], labels[:, 1:])
    np.fill_diagonal(iou, 0)
    return iou.max(1)

boxes["max_iou"] = np.concatenate([max_overlap(read_labels(iid)) for iid in boxes["image_id"].unique()])
for t in (0.1, 0.3, 0.5):
    print(f"{(boxes['max_iou'] > t).mean():5.1%} of the boxes overlap another one with IoU > {t}")

**Questions**
- mAP averages the AP of the 7 classes. Which classes have so few boxes that one or two
  misses move the score? What does that mean for the fish, the bulk of the boxes?
- Most boxes are small. A detector that looks at the image through a grid of 7 × 7 cells
  (64 px each at 448) puts how many fish in one cell?
- NMS deletes a box when it overlaps a better box of the same class above a threshold. With
  a threshold of 0.5, which true boxes could it delete?

## 5. A small YOLO from scratch

Our detector has two parts:

- a **backbone**, ResNet-18 trained to *classify* ImageNet: it turns the image into a map
  of features (edges, textures, shapes) but knows nothing about boxes;
- a **head**, trained from scratch here, that cuts the image into an **S × S grid**. Each
  cell has **B anchors** (box shapes it starts from), and each anchor predicts 5 + C numbers:

| output | meaning | decoded as |
|---|---|---|
| $t_x, t_y$ | centre of the box inside the cell | $x = (\text{col} + \sigma(t_x)) / S$ |
| $t_w, t_h$ | size, relative to the anchor | $w = w_{anchor} \cdot e^{t_w}$ |
| $t_o$ | is there an object? | $\sigma(t_o)$ |
| C logits | which class, if there is one | softmax |

The output is a tensor $S \times S \times B \times (5 + C)$. S, B and the anchor shapes are
the three hyperparameters of the head.

### 5.1 Anchors: typical box shapes

Anchors are chosen by **k-means** on the (width, height) of the training boxes: B clusters,
B typical shapes.

In [ ]:
import torch.nn as nn, torch.nn.functional as F, torchvision
from sklearn.cluster import KMeans

C, IMG = len(class_names), 448                 # images are resized to IMG x IMG
dev = "cuda" if torch.cuda.is_available() else "cpu"
train_wh = boxes[["w", "h"]].to_numpy()

def kmeans_anchors(B):
    centers = KMeans(n_clusters=B, n_init=10, random_state=0).fit(train_wh).cluster_centers_
    return torch.tensor(centers, dtype=torch.float32)

plt.figure(figsize=(5, 5))
plt.scatter(train_wh[:, 0], train_wh[:, 1], s=2, alpha=0.3)
for B, marker in ((1, "o"), (3, "s"), (5, "^")):
    a = kmeans_anchors(B)
    plt.scatter(a[:, 0], a[:, 1], s=120, marker=marker, label=f"B = {B}")
plt.xlabel("width"); plt.ylabel("height"); plt.legend(); plt.title("training boxes and k-means anchors"); plt.show()

### 5.2 From labels to targets

Each true box is given to **one** cell (the one holding its centre) and **one** anchor
(the closest in shape). If that slot is already taken, the box is dropped: the model is
never asked to find it.

In [ ]:
def shape_iou(wh, anchors):
    """IoU between a box (w, h) and each anchor, all centred on the same point."""
    inter = torch.minimum(wh[0], anchors[:, 0]) * torch.minimum(wh[1], anchors[:, 1])
    return inter / (wh[0] * wh[1] + anchors[:, 0] * anchors[:, 1] - inter)

def encode(labels, S, anchors):
    """Labels (n, 5) -> target S x S x B x 6: x, y in the cell, log(w / anchor), log(h / anchor), object, class."""
    target, dropped = torch.zeros(S, S, len(anchors), 6), 0
    for c, x, y, w, h in torch.tensor(labels, dtype=torch.float32):
        col, row = min(int(x * S), S - 1), min(int(y * S), S - 1)
        a = shape_iou(torch.stack([w, h]), anchors).argmax()
        if target[row, col, a, 4] == 1:
            dropped += 1
            continue
        target[row, col, a] = torch.stack([x * S - col, y * S - row, torch.log(w / anchors[a, 0]),
                                           torch.log(h / anchors[a, 1]), torch.tensor(1.0), c])
    return target, dropped

Y_train = [read_labels(iid) for iid in split["train"]]
dropped = {}
for B in (1, 3, 5):
    anchors = kmeans_anchors(B)
    for S in (7, 14):
        dropped[f"B = {B}", f"S = {S}"] = sum(encode(y, S, anchors)[1] for y in Y_train) / len(boxes)
pd.Series(dropped).unstack().style.format("{:.0%}").set_caption("training boxes dropped")

**Question:** a dropped box is a box the model can never find. What is the best recall a
model with S = 7, B = 1 can reach on this data, and which hyperparameter fixes it faster?

### 5.3 The model and its loss

In [ ]:
class SmallYOLO(nn.Module):
    def __init__(self, S, B, pretrained_backbone=True):
        super().__init__()
        resnet = torchvision.models.resnet18(weights="IMAGENET1K_V1" if pretrained_backbone else None)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])     # 512 x IMG/32 x IMG/32
        self.head = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.1),
            nn.AdaptiveAvgPool2d(S),                                       # the S x S grid
            nn.Conv2d(256, B * (5 + C), 1))                                # B x (5 + C) numbers per cell
        self.S, self.B = S, B
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1))

    def forward(self, images):                                             # uint8 N x 3 x IMG x IMG
        x = (images.float() / 255 - self.mean) / self.std
        out = self.head(self.backbone(x))                                  # N x B(5+C) x S x S
        return out.permute(0, 2, 3, 1).reshape(len(x), self.S, self.S, self.B, 5 + C)

Five terms, as in the *YOLO* lesson. The slots that hold a box learn its position, its size,
"there is an object" and its class; **every other slot** only learns "no object".

In [ ]:
def yolo_loss(out, target, l_box=5.0, l_noobj=0.5):
    has_obj = target[..., 4] == 1
    p, t = out[has_obj], target[has_obj]           # the slots responsible for a true box
    empty = out[~has_obj][:, 4]                    # objectness of all the other slots
    terms = {
        "xy":    l_box * F.mse_loss(p[:, :2].sigmoid(), t[:, :2], reduction="sum"),
        "wh":    l_box * F.mse_loss(p[:, 2:4], t[:, 2:4], reduction="sum"),
        "obj":   F.binary_cross_entropy_with_logits(p[:, 4], torch.ones_like(p[:, 4]), reduction="sum"),
        "noobj": l_noobj * F.binary_cross_entropy_with_logits(empty, torch.zeros_like(empty), reduction="sum"),
        "cls":   F.cross_entropy(p[:, 5:], t[:, 5].long(), reduction="sum"),
    }
    return {name: value / len(out) for name, value in terms.items()}   # per image

### 5.4 Train

In [ ]:
def load_images(paths):
    """uint8 tensor N x 3 x IMG x IMG. Squashing to a square keeps the normalised labels valid."""
    return torch.stack([torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((IMG, IMG)))).permute(2, 0, 1)
                        for p in paths])

X_train = load_images([f"data/train/images/{iid}.jpg" for iid in split["train"]])
X_val = load_images([f"data/train/images/{iid}.jpg" for iid in split["val"]])
val_gt = {iid: read_labels(iid) for iid in split["val"]}

def train_small_yolo(S=7, B=1, anchors=None, epochs=30, pretrained_backbone=True, **loss_weights):
    torch.manual_seed(0)
    anchors = kmeans_anchors(B) if anchors is None else anchors
    model = SmallYOLO(S, B, pretrained_backbone).to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    val_target = torch.stack([encode(val_gt[iid], S, anchors)[0] for iid in split["val"]])
    history = []
    for epoch in range(epochs):
        model.train()
        logged = {}
        for batch in torch.randperm(len(X_train)).split(16):
            images, targets = [], []
            for i in batch.tolist():
                image, labels = X_train[i], Y_train[i].copy()
                if random.random() < 0.5:                       # augmentation: horizontal flip
                    image, labels[:, 1] = image.flip(-1), 1 - labels[:, 1]
                images.append(image); targets.append(encode(labels, S, anchors)[0])
            terms = yolo_loss(model(torch.stack(images).to(dev)), torch.stack(targets).to(dev), **loss_weights)
            optimizer.zero_grad(); sum(terms.values()).backward(); optimizer.step()
            for name, value in terms.items():
                logged[f"train {name}"] = logged.get(f"train {name}", 0) + value.item() * len(batch) / len(X_train)
        model.eval()
        with torch.no_grad():
            out = torch.cat([model(x.to(dev)).cpu() for x in X_val.split(16)])
        logged.update({f"val {name}": value.item() for name, value in yolo_loss(out, val_target, **loss_weights).items()})
        history.append(logged)
    return model, anchors, pd.DataFrame(history)

small, small_anchors, history = train_small_yolo(S=7, B=1)
trained = {"S=7 B=1": (small, small_anchors)}

terms = ["xy", "wh", "obj", "noobj", "cls"]
fig, axes = plt.subplots(1, 5, figsize=(20, 3))
for ax, name in zip(axes, terms):
    ax.plot(history[f"train {name}"], label="train"); ax.plot(history[f"val {name}"], label="val")
    ax.set_title(name); ax.set_xlabel("epoch")
axes[0].legend(); plt.tight_layout(); plt.show()

**Questions**
- Which term is the largest at the first epoch, and why? Count the slots: how many hold an
  object, how many are empty?
- Which terms keep falling on train but not on validation? What is the model learning by heart?
- Why is `noobj` weighted by 0.5 and the box terms by 5?

### 5.5 Decode and analyse the predictions

Decoding turns the tensor back into boxes: score = objectness × class probability, then
**NMS** removes the duplicates. For mAP we keep every box down to a confidence of 0.001.

In [ ]:
def decode(out, anchors, conf=0.001, nms_iou=0.5, max_det=300):
    """One image's S x S x B x (5 + C) output -> (n, 6) array: class, score, x, y, w, h."""
    S = out.shape[0]
    row, col = torch.meshgrid(torch.arange(S), torch.arange(S), indexing="ij")
    x = (col[..., None] + out[..., 0].sigmoid()) / S
    y = (row[..., None] + out[..., 1].sigmoid()) / S
    w = anchors[:, 0] * out[..., 2].clamp(max=4).exp()
    h = anchors[:, 1] * out[..., 3].clamp(max=4).exp()
    xywh = torch.stack([x, y, w, h], -1).reshape(-1, 4)
    scores = (out[..., 4:5].sigmoid() * out[..., 5:].softmax(-1)).reshape(-1, C)
    box, cls = (scores > conf).nonzero(as_tuple=True)                  # every (box, class) above conf
    score, xywh = scores[box, cls], xywh[box]
    keep = torchvision.ops.batched_nms(torchvision.ops.box_convert(xywh, "cxcywh", "xyxy"), score, cls, nms_iou)
    keep = keep[:max_det]
    return torch.column_stack([cls[keep].float(), score[keep], xywh[keep]]).numpy()

def predict_small(model, anchors, images, image_ids):
    model.eval()
    with torch.no_grad():
        out = torch.cat([model(x.to(dev)).cpu() for x in images.split(16)])
    return {iid: decode(o, anchors) for iid, o in zip(image_ids, out)}

small_pred = predict_small(small, small_anchors, X_val, split["val"])

One analysis for every model of this notebook: the score, then **which true boxes are found**
(a prediction of the same class with IoU ≥ 0.5 and confidence ≥ 0.25), by class and by size.

In [ ]:
def analyse(pred, conf=0.25):
    s = dm.evaluate(val_gt, pred)
    print(f"validation mAP50-95 {s['map50_95']:.3f}   mAP50 {s['map50']:.3f}")
    rows = []
    for iid, truth in val_gt.items():
        p = pred[iid][pred[iid][:, 1] >= conf]
        iou = dm.box_iou(truth[:, 1:], p[:, 2:]) * (truth[:, :1] == p[:, 0])     # same class only
        found = iou.max(1, initial=0) >= 0.5
        rows += [(class_names[int(c)], np.sqrt(w * h), f) for (c, x, y, w, h), f in zip(truth, found)]
    found = pd.DataFrame(rows, columns=["class", "size", "found"])
    found["size"] = pd.cut(found["size"], [0, 0.05, 0.15, 1], labels=["small", "medium", "large"])
    by_class = pd.DataFrame({"AP50-95": {c: s.get(f"ap_{c}", 0) for c in class_names},
                             "recall": found.groupby("class")["found"].mean()})
    display(by_class.T.round(2))
    display(found.groupby("size", observed=True)["found"].mean().round(2).to_frame("recall").T)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for col, iid in enumerate(split["val"][:3]):
        draw(axes[0, col], f"yolo/val/images/{iid}.jpg", val_gt[iid], "truth")
        draw(axes[1, col], f"yolo/val/images/{iid}.jpg", pred[iid][pred[iid][:, 1] >= conf], f"predicted, confidence >= {conf}")
    plt.tight_layout(); plt.show()
    return s["map50_95"]

results = {"small YOLO S=7 B=1": analyse(small_pred)}

**Questions**
- Recall by size: does it match what section 4 predicted for a 7 × 7 grid?
- The pictures show boxes with confidence ≥ 0.25, the score uses all of them down to 0.001.
  Why can extra low-confidence boxes raise mAP but never lower it?
- mAP50 against mAP50-95: are the boxes in the right place, or also tight?

### 5.6 Play with the head

Change **one** hyperparameter at a time and keep a table. Each run takes a few minutes.
Before each run, write down what you expect, from 5.2 and from the analysis above.

In [ ]:
for S, B in [(14, 1), (14, 3)]:                              # try also (7, 3), (14, 5), ...
    model, anchors, _ = train_small_yolo(S=S, B=B)
    trained[f"S={S} B={B}"] = (model, anchors)
    results[f"small YOLO S={S} B={B}"] = dm.evaluate(val_gt, predict_small(model, anchors, X_val, split["val"]))["map50_95"]
pd.Series(results, name="validation mAP50-95").round(3)

More to try, one at a time:
- `(14, 5)`: five anchors instead of three. Can more anchors make the model worse? Count the
  empty slots in the `noobj` term, and the boxes each anchor learns from.
- `anchors=torch.tensor([[0.15, 0.15]] * 3)`: three identical square anchors instead of k-means.
  What happens to the slots, and to the score?
- `pretrained_backbone=False`: the whole network from scratch, ImageNet included.
- `l_noobj=5.0` or `l_noobj=0.05`: watch the confidences in the pictures.

### 5.7 First submission

Take your best setting, look at its predictions with `analyse`, then predict the 190 test
images and submit.

In [ ]:
def submit(pred, name):
    """pred: {image_id: (n, 6) array}. Writes submission.csv and sends it to ML-Arena."""
    rows = [{"image_id": iid, "prediction_string": dm.to_prediction_string(b)} for iid, b in pred.items()]
    pd.DataFrame(rows).to_csv("submission.csv", index=False)
    print(client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"], submission_name=name))

best = "S=14 B=3"                                             # your best setting
small, small_anchors = trained[best]
analyse(predict_small(small, small_anchors, X_val, split["val"]))
X_test = load_images(test_images)
submit(predict_small(small, small_anchors, X_test, [p.stem for p in test_images]), f"small-yolo-{best.replace(' ', '-').replace('=', '')}")

## 6. Pretrained detectors, as they are

Ultralytics publishes detectors already trained on large datasets. Three small ones:

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
pretrained = {"yolov8n.pt": "COCO", "yolo11n.pt": "COCO", "yolov8n-oiv7.pt": "Open Images V7"}
models = {name: YOLO(name) for name in pretrained}

pd.DataFrame({name: {"trained on": pretrained[name],
                     "parameters (M)": round(sum(p.numel() for p in m.model.parameters()) / 1e6, 1),
                     "classes": len(m.names),
                     "our classes it knows": [c for c in class_names if c in {n.lower() for n in m.names.values()}],
                     "first classes": ", ".join(list(m.names.values())[:8]) + ", ..."}
              for name, m in models.items()}).T

Apply them to a validation image. `predict` returns, for every box, a class index **in the
model's own list**, a confidence and the coordinates.

In [ ]:
iid = split["val"][0]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
draw(axes[0], f"yolo/val/images/{iid}.jpg", val_gt[iid], "truth")
for ax, (name, m) in zip(axes[1:], models.items()):
    r = m.predict(f"yolo/val/images/{iid}.jpg", conf=0.25, device=device, verbose=False)[0]
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

r = models["yolov8n.pt"].predict(f"yolo/val/images/{iid}.jpg", conf=0.25, device=device, verbose=False)[0]
print(r.boxes.cls, r.boxes.conf, r.boxes.xywhn, sep="\n")

Over the whole validation set: for each true box, the best-overlapping prediction (IoU ≥ 0.5,
**whatever its class**), and the name the model gave it.

In [ ]:
def what_it_sees(m, conf=0.25):
    rows = []
    for iid, truth in val_gt.items():
        r = m.predict(f"yolo/val/images/{iid}.jpg", conf=conf, device=device, verbose=False)[0]
        iou = dm.box_iou(truth[:, 1:], r.boxes.xywhn.cpu().numpy())
        for k, c in enumerate(truth[:, 0]):
            j = iou[k].argmax() if iou.shape[1] else None
            seen = m.names[int(r.boxes.cls[j])] if j is not None and iou[k, j] >= 0.5 else "(missed)"
            rows.append((class_names[int(c)], seen))
    return pd.DataFrame(rows, columns=["truth", "predicted as"])

for name, m in models.items():
    seen = what_it_sees(m)
    print(f"{name}: {(seen['predicted as'] != '(missed)').mean():.0%} of the true boxes are covered by a box")
    table = pd.crosstab(seen["truth"], seen["predicted as"])
    display(table[table.sum().sort_values(ascending=False).index[:8]])   # the 8 most frequent answers

**Questions**
- COCO has no fish. What does a COCO model call them, and does it at least put a box in the
  right place?
- The Open Images model knows "Fish", "Jellyfish", "Shark". Is that enough to score well
  here? Think of a school of fish: one box per fish, or one for the school?
- Fine-tuning keeps the backbone and the box regression and replaces the class layer
  (80 or 600 outputs) by one with 7. From the tables above, what part of a pretrained
  detector is already useful for us, and what has to be learnt?

## 7. Fine-tune one

Pick one of the three. Ultralytics swaps the class layer for our 7 classes and trains all
the weights on our data (about 5–10 minutes on a T4).

In [ ]:
MODEL = "yolov8n.pt"                                          # or "yolo11n.pt", "yolov8n-oiv7.pt"
model = YOLO(MODEL)
model.train(data="aquarium.yaml", epochs=30, imgsz=640, batch=16, seed=0, device=device)
model = YOLO(Path(model.trainer.save_dir) / "weights" / "best.pt")   # best validation epoch

The same analysis as for the small YOLO.

In [ ]:
def predict_yolo(m, paths, conf=0.001):
    """{image_id: (n, 6) array}: class, confidence, x, y, w, h (normalised as the labels)."""
    pred = {}
    for i in range(0, len(paths), 16):
        batch = paths[i:i + 16]
        for p, r in zip(batch, m.predict([str(p) for p in batch], conf=conf, device=device, verbose=False)):
            b = r.boxes
            pred[Path(p).stem] = np.column_stack([b.cls.cpu().numpy(), b.conf.cpu().numpy(), b.xywhn.cpu().numpy()]).reshape(-1, 6)
    return pred

val_images = [Path(f"yolo/val/images/{iid}.jpg") for iid in split["val"]]
results[f"fine-tuned {MODEL}"] = analyse(predict_yolo(model, val_images))
pd.Series(results, name="validation mAP50-95").round(3)

**Questions**
- Against the best small YOLO: which classes and which box sizes gained the most?
- The fine-tuned model saw the same 381 images as the small YOLO. Where does the gap come
  from: the backbone, the head (three strides, see the lesson), the augmentation, the
  pretraining on boxes?

Submit it:

In [ ]:
submit(predict_yolo(model, test_images), f"finetuned-{MODEL.removesuffix('.pt')}")

## Going further

Change one thing, compare the validation mAP50-95, submit only if it improved: more epochs,
a bigger model (`yolov8s.pt`, `yolo11s.pt`), `imgsz=800`, retraining on all 448 images once
the settings are chosen.